# 01 — Stacking from scratch

Implémentation maison d’un `CustomStackingClassifier` avec Out-of-Fold predictions.

In [ ]:
# ============================================================
# Setup Colab — AML Stacking & Blending
# ============================================================
import sys, subprocess, pkgutil, warnings, random, time, os
warnings.filterwarnings("ignore")

def install_if_missing(package, import_name=None):
    import_name = import_name or package.replace("-", "_")
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, import_name in [
    ("numpy","numpy"), ("pandas","pandas"), ("matplotlib","matplotlib"),
    ("seaborn","seaborn"), ("scikit-learn","sklearn"), ("mlxtend","mlxtend"),
    ("xgboost","xgboost"), ("lightgbm","lightgbm"), ("vecstack","vecstack")
]:
    install_if_missing(package, import_name)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from mlxtend.classifier import StackingCVClassifier
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
# ============================================================
# Dataset principal — UCI Wine Quality
# ============================================================
UCI_RED_WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(UCI_RED_WINE_URL, sep=";")

# Classification binaire : bonne qualité si quality >= 6
X = df.drop(columns=["quality"])
y = (df["quality"] >= 6).astype(int)

display(df.head())
display(pd.DataFrame({"target": y}).value_counts(normalize=True).rename("proportion"))
display(X.describe().T)

plt.figure(figsize=(5,3))
sns.countplot(x=y)
plt.title("Distribution de la cible : quality >= 6")
plt.show()

In [ ]:
# ============================================================
# Prétraitement et modèles
# ============================================================
def build_preprocessor(X):
    num_cols = X.select_dtypes(include=["int64","float64","int32","float32"]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object","category","bool"]).columns.tolist()
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

def wrap(model):
    return Pipeline([("prep", build_preprocessor(X)), ("model", model)])

def get_base_learners(seed=42):
    learners = [
        ("lr", wrap(LogisticRegression(max_iter=5000, random_state=seed))),
        ("svm", wrap(SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, random_state=seed))),
        ("knn", wrap(KNeighborsClassifier(n_neighbors=9))),
        ("rf", wrap(RandomForestClassifier(n_estimators=250, random_state=seed, n_jobs=-1))),
        ("gb", wrap(GradientBoostingClassifier(random_state=seed))),
    ]
    if XGBClassifier is not None:
        learners.append(("xgb", wrap(XGBClassifier(n_estimators=250, learning_rate=0.05, max_depth=4, eval_metric="logloss", random_state=seed))))
    if LGBMClassifier is not None:
        learners.append(("lgbm", wrap(LGBMClassifier(n_estimators=250, learning_rate=0.05, random_state=seed, verbose=-1))))
    return learners

def evaluate(model, X_train, y_train, X_test, y_test, name):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    pred = model.predict(X_test)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "train_time_sec": elapsed,
    }
    if hasattr(model, "predict_proba"):
        try:
            row["roc_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        except Exception:
            row["roc_auc"] = np.nan
    return row

In [ ]:
# ============================================================
# CustomStackingClassifier — from scratch
# ============================================================
class CustomStackingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimators, meta_estimator=None, n_splits=5, random_state=42):
        self.base_estimators = base_estimators
        self.meta_estimator = meta_estimator
        self.n_splits = n_splits
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        self.names_ = [name for name, _ in self.base_estimators]
        Z = np.zeros((len(X), len(self.base_estimators)))
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

        for fold_id, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
            print(f"OOF fold {fold_id}/{self.n_splits}")
            for j, (name, estimator) in enumerate(self.base_estimators):
                model = clone(estimator)
                model.fit(X.iloc[train_idx], y.iloc[train_idx])
                Z[valid_idx, j] = model.predict_proba(X.iloc[valid_idx])[:, 1]

        self.oof_meta_features_ = pd.DataFrame(Z, columns=self.names_)
        self.meta_estimator_ = clone(self.meta_estimator or LogisticRegression(max_iter=5000, random_state=self.random_state))
        self.meta_estimator_.fit(self.oof_meta_features_, y)

        self.fitted_base_estimators_ = []
        for name, estimator in self.base_estimators:
            model = clone(estimator)
            model.fit(X, y)
            self.fitted_base_estimators_.append((name, model))
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        Z = np.zeros((len(X), len(self.fitted_base_estimators_)))
        for j, (name, model) in enumerate(self.fitted_base_estimators_):
            Z[:, j] = model.predict_proba(X)[:, 1]
        return pd.DataFrame(Z, columns=self.names_)

    def predict(self, X):
        return self.meta_estimator_.predict(self.transform(X))

    def predict_proba(self, X):
        return self.meta_estimator_.predict_proba(self.transform(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=SEED, stratify=y)
model = CustomStackingClassifier(get_base_learners(SEED), n_splits=5, random_state=SEED)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))
print("F1 macro:", f1_score(y_test, pred, average="macro"))
display(model.oof_meta_features_.head())